# Package

In [15]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error

import mlflow

# Importation des données

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
from pathlib import Path
import pandas as pd
from feast import FeatureStore

def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())

# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(
        entity_df=entity_df,
        features=feature_refs,
        full_feature_names=True,
    ).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]

# On veut les séries stationnarisées pour le modèle
FEATURE_REFS = ["stationary_value:value"]

# ----------------------------
# 1) Dates de référence (via UNRATE raw) pour avoir le vrai calendrier dispo
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})
df_unrate_dates = load_features_from_feast(entity_df_unrate, ["raw_value:value"])

dates = (
    pd.to_datetime(df_unrate_dates["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

# ----------------------------
# 2) Entity DF multi-séries × dates (long)
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

# ----------------------------
# 3) Fetch stationary features (long)
# ----------------------------
df_stationary = load_features_from_feast(entity_df, FEATURE_REFS)

df_stationary["date"] = (
    pd.to_datetime(df_stationary["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
)

value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Expected '{value_col}' not found. Candidates: {candidates}")

df_stationary = (
    df_stationary[["series_id", "date", value_col]]
    .rename(columns={value_col: "value"})
)

print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

# ----------------------------
# 4) Dataset régression ciblé UNRATE
#    y = UNRATE (stationary)
#    exog = autres séries (contemporaines)
# ----------------------------
df_y = (
    df_stationary[df_stationary["series_id"] == "UNRATE"]
    .sort_values("date")
    .rename(columns={"value": "y"})
    .reset_index(drop=True)
)

df_x_long = df_stationary[df_stationary["series_id"] != "UNRATE"].copy()
df_x_long = df_x_long[df_x_long["date"].isin(df_y["date"])]

df_x = (
    df_x_long
    .pivot_table(index="date", columns="series_id", values="value", aggfunc="last")
    .reset_index()
)

df_model = (
    df_y[["date", "y"]]
    .merge(df_x, on="date", how="left")
    .dropna()
)

# ----------------------------
# 5) Format MLForecast (long + exog cols)
# ----------------------------
ts_lr = df_model.rename(columns={"date": "ds"})
ts_lr["unique_id"] = "UNRATE"

exog_cols = [c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]]
ts_lr = ts_lr[["unique_id", "ds", "y"] + exog_cols]

print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols)
ts_lr.head()

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date     value
0  BUSLOANS 1960-01-01  0.011578
1    INDPRO 1960-01-01  0.091976
2     USREC 1960-01-01  0.000000
3      M2SL 1960-01-01  0.001323
4  CPIAUCSL 1960-01-01 -0.006156
ts_lr shape: (788, 13)
Exog cols: ['BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX', 'RPI', 'SP500', 'TB3MS', 'USREC']


,unique_id,ds,y,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,USREC
0,UNRATE,1960-01-01,-0.8,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,0.0
1,UNRATE,1960-02-01,-1.1,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,0.0
2,UNRATE,1960-03-01,-0.2,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,0.0
3,UNRATE,1960-04-01,0.0,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0
4,UNRATE,1960-05-01,0.0,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,1.0


In [3]:
ts_lr["ds"] = (
    pd.to_datetime(ts_lr["ds"])
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# AR

In [4]:
y = ts_lr[["ds", "y"]]
y.index = y.ds
y.index.name = "date"

In [5]:
y = y.drop(["ds"], axis = 1)
y.head()

,y
date,
1960-01-01,-0.8
1960-02-01,-1.1
1960-03-01,-0.2
1960-04-01,0.0
1960-05-01,0.0


In [6]:
# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

In [7]:

# ---------- Paramètres ----------
h = 12
min_train_n = 36
trend = "c"
p_fixed = 1

# ---------- Paramètres bagging ----------
use_bagging = True
B_boot = 30
L_block = 12
rng = np.random.default_rng(123)

# ---------- Paramètres conformal (comme code 1) ----------
use_conformal = True
step_size = 12
pi_windows = 3
alpha = 0.05  # 95% => q = quantile 1-alpha des erreurs

In [8]:
# =========================
# cellule 2 : Bootstrap + fonctions Conformal
# =========================
def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à longueur n."""
    n = len(arr)
    if L <= 0 or L > n:
        raise ValueError("L_block invalide")
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def bagged_h_forecast_AR1(y_tr, h, trend, B, L, rng):
    """
    Prévision à horizon h par bagging (residual moving-block bootstrap) pour AR(1).
    Retourne (yhat_mean, yhat_dist, base_pred)
    """
    base_model = AutoReg(y_tr, lags=1, old_names=False, trend=trend).fit()
    base_fc = base_model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    base_pred = float(base_fc.iloc[-1])

    resid = base_model.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné aux résidus

    boot_preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)
        y_b = fitted + res_b
        m_b = AutoReg(pd.Series(y_b, index=y_tr.index[-len(y_b):]),
                      lags=1, old_names=False, trend=trend).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        boot_preds.append(float(fc_b.iloc[-1]))

    return float(np.mean(boot_preds)), np.array(boot_preds), base_pred

def fit_predict_ar_p(y_tr, h, trend="c", p=1):
    """Fit AR(p) sur y_tr, retourne la prévision au pas h (dernier point)."""
    m = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
    fc = m.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    return float(fc.iloc[-1])

def conformal_q_from_past_windows_pos(y, i_end, *, h=12, step_size=12, pi_windows=3, trend="c", p=1, alpha=0.05):
    """
    Version robuste (sans test 'date in index'):
    - i_end = position de t_end dans y.index
    - fenêtres de calibration: i_cal = i_end - k*step_size
    - cible à comparer: i_cal + h
    - erreurs: |y[i_cal+h] - yhat( train jusqu'à i_cal )|
    """
    errs = []
    for k in range(1, pi_windows + 1):
        i_cal = i_end - k * step_size
        i_cal_fore = i_cal + h
        if i_cal < 0 or i_cal_fore >= len(y):
            continue

        y_tr_cal = y.iloc[: i_cal + 1]
        if len(y_tr_cal) < max(36, p + 2):  # cohérent avec min_train_n
            continue

        yhat_cal = fit_predict_ar_p(y_tr_cal, h=h, trend=trend, p=p)
        err = abs(float(y.iloc[i_cal_fore]) - yhat_cal)
        errs.append(err)

    if len(errs) == 0:
        return np.nan

    # intervalle symétrique: q = quantile(1-alpha) des erreurs absolues
    return float(np.quantile(errs, 1 - alpha))

In [9]:
# =========================
# cellule 3 : Sécurisation de la série y (index strictement début de mois)
# =========================
y = pd.Series(y.astype(float).values, index=pd.to_datetime(y.index))

# Aligne sur le 1er du mois (start of month) de façon robuste
y.index = y.index.to_period("M").to_timestamp(how="start")

# Fixe la fréquence MS (Month Start)
y = y.asfreq("MS").dropna()

print(f"y: {y.index.min().date()} → {y.index.max().date()}  (n={len(y)}) | freq={y.index.freqstr}")

y: 1960-01-01 → 2025-08-01  (n=788) | freq=MS


In [10]:
# =========================
# cellule 4 : Boucle pseudo-OOS continue + prédiction + intervalles (conformal ou bootstrap)
# =========================
rows = []
last_model = None
last_fit_end = None

# on boucle uniquement sur des t_end qui ont un t_end+h existant
for i_end, t_end in enumerate(y.index[:-h]):
    y_tr = y.iloc[: i_end + 1]
    if len(y_tr) < max(min_train_n, p_fixed + 1):
        continue

    # fit AR(p) base
    ar1 = AutoReg(y_tr, lags=p_fixed, old_names=False, trend=trend).fit()
    last_model = ar1
    last_fit_end = t_end

    # ----- Prévision à h mois (bagging ou base) -----
    if use_bagging:
        yhat_h, yhat_dist, yhat_h_base = bagged_h_forecast_AR1(
            y_tr=y_tr, h=h, trend=trend, B=B_boot, L=L_block, rng=rng
        )
    else:
        fc = ar1.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        yhat_h_base = yhat_h
        yhat_dist = None

    # t_fore pris DIRECTEMENT depuis l'index (robuste)
    t_fore = y.index[i_end + h]
    y_true = float(y.iloc[i_end + h])

    # ----- Intervalles -----
    if use_conformal:
        q = conformal_q_from_past_windows_pos(
            y=y, i_end=i_end, h=h, step_size=step_size, pi_windows=pi_windows,
            trend=trend, p=p_fixed, alpha=alpha
        )
        if np.isfinite(q):
            yhat_p05 = yhat_h - q
            yhat_p95 = yhat_h + q
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan
    else:
        # fallback: quantiles bootstrap (si bagging), sinon NA
        if use_bagging and (yhat_dist is not None) and len(yhat_dist) > 0:
            yhat_p05 = float(np.percentile(yhat_dist, 5))
            yhat_p95 = float(np.percentile(yhat_dist, 95))
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan

    rows.append((t_fore, yhat_h, y_true, yhat_p05, yhat_p95, yhat_h_base))

In [11]:
# =========================
# cellule 5 : DataFrame OOS
# =========================
if rows:
    df_oos_ar1 = (
        pd.DataFrame(
            rows,
            columns=["date", "y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"]
        )
        .set_index("date")
        .sort_index()
    )
else:
    df_oos_ar1 = pd.DataFrame(columns=["y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"])
    df_oos_ar1.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_ar1)}")
print(df_oos_ar1.head(-3))


✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  y_hat_p05  y_hat_p95  y_hat_base
date                                                          
1963-12-01  0.070473     0.0        NaN        NaN   -0.080890
1964-01-01  0.017682    -0.1        NaN        NaN    0.141077
1964-02-01  0.090722    -0.5        NaN        NaN    0.408114
1964-03-01  0.165681    -0.3        NaN        NaN    0.242637
1964-04-01  0.091963    -0.4        NaN        NaN    0.238955
...              ...     ...        ...        ...         ...
2025-01-01 -0.010382     0.3  -3.300127   3.279363    0.043861
2025-02-01 -0.006875     0.2  -3.275618   3.261868    0.072590
2025-03-01 -0.019816     0.3  -2.895811   2.856179    0.101443
2025-04-01  0.005002     0.3  -0.589461   0.599465    0.130432
2025-05-01 -0.003852     0.2  -0.617284   0.609580    0.102350

[738 rows x 5 columns]


In [12]:
params = {
    "model": "AR(1)",
    "trend": trend,
    "horizon": h,
    "lags": p_fixed,
    "min_train_n": min_train_n,
    "use_bagging": bool(use_bagging),
    "B_boot": int(B_boot) if use_bagging else None,
    "L_block": int(L_block) if use_bagging else None,
    "use_conformal": bool(use_conformal),
    "pi_windows": int(pi_windows) if use_conformal else None,
    "step_size": int(step_size) if use_conformal else None,
    "alpha": float(alpha) if use_conformal else None,
}

In [13]:
# --- Metrics (résultats numériques) ---
# ex: MAE global
df_eval = df_oos_ar1.dropna(subset=["y_true", "y_hat"])
err = df_eval["y_true"] - df_eval["y_hat"]
mae = float(np.mean(np.abs(err)))

# coverage 95% si dispo
df_pi = df_oos_ar1.dropna(subset=["y_hat_p05", "y_hat_p95", "y_true"])
if len(df_pi) > 0:
    coverage = float(((df_pi["y_true"] >= df_pi["y_hat_p05"]) & (df_pi["y_true"] <= df_pi["y_hat_p95"])).mean())
else:
    coverage = np.nan

metrics = {
    "mae_all": mae,
    "pi_coverage_95": coverage,
    "n_forecasts": int(len(df_oos_ar1)),
}

In [ ]:
# -------------------------------------------------
# 1) Définition des périodes
# -------------------------------------------------
periods = {
    "ALL":        (None, None),
    "1990_1999":  ("1990-01-01", "1999-12-31"),
    "2000_2008":  ("2000-01-01", "2008-12-31"),
    "2008_2019":  ("2008-01-01", "2019-12-31"),
    "2020_fin":   ("2020-01-01", None),
}

# -------------------------------------------------
# 2) Calcul métriques
# -------------------------------------------------
df = df_oos_ar1.copy().sort_index()
df.index = pd.to_datetime(df.index)

metrics = {}

for label, (start, end) in periods.items():

    sub = df.copy()

    if start is not None:
        sub = sub[sub.index >= pd.Timestamp(start)]
    if end is not None:
        sub = sub[sub.index <= pd.Timestamp(end)]

    if len(sub) == 0:
        continue

    mae = mean_absolute_error(sub["y_true"], sub["y_hat"])

    # métriques MLflow
    metrics[f"mae_{label}"] = float(mae)
    metrics[f"n_obs_{label}"] = int(len(sub))

In [17]:
metrics

{'mae_ALL': 0.8787949660045873,
 'n_obs_ALL': 741,
 'mae_1990_1999': 0.5077934088921013,
 'n_obs_1990_1999': 120,
 'mae_2000_2008': 0.535631096135122,
 'n_obs_2000_2008': 108,
 'mae_2008_2019': 0.8685891995750803,
 'n_obs_2008_2019': 144,
 'mae_2020_fin': 2.0848489059737685,
 'n_obs_2020_fin': 68}

In [18]:
last_model

In [21]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Test_MLFLOW")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1771180569031, experiment_id='1', last_update_time=1771180569031, lifecycle_stage='active', name='Test_MLFLOW', tags={}>

In [ ]:
import mlflow
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("Test_MLFLOW"))

In [22]:
import mlflow
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("Test_MLFLOW"))

Tracking URI: http://127.0.0.1:5000
Experiment: <Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1771180569031, experiment_id='1', last_update_time=1771180569031, lifecycle_stage='active', name='Test_MLFLOW', tags={}>


In [53]:
import mlflow
import pickle
from pathlib import Path

# -----------------------------
# 0) Nom du modèle
# -----------------------------
MODEL_NAME = "Autoregressive_1"

# -----------------------------
# 1) Initialize MLflow
# -----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Test_MLFLOW")

run_name = f"{MODEL_NAME}_h{h}_bag{int(use_bagging)}_conf{int(use_conformal)}"

# -----------------------------
# 2) Artifact path (modèle)
# -----------------------------
TMP_DIR = Path("outputs/tmp_models")
TMP_DIR.mkdir(parents=True, exist_ok=True)
model_path = TMP_DIR / f"{MODEL_NAME}_last_model.pkl"

# Cohérence dans params
params["model"] = MODEL_NAME

with mlflow.start_run(run_name=run_name):

    # -----------------------------
    # Tags
    # -----------------------------
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("end", str(last_fit_end.date()) if last_fit_end is not None else "unknown")
    mlflow.set_tag("freq", y.index.freqstr if y.index.freq is not None else "MS")

    # -----------------------------
    # Params
    # -----------------------------
    for k, v in params.items():
        if v is not None:
            mlflow.log_param(k, v)

    # -----------------------------
    # Metrics (MAE uniquement)
    # n_obs en tags
    # -----------------------------
    for k, v in metrics.items():

        if v is None:
            continue

        if k.startswith("mae_"):
            mlflow.log_metric(k, float(round(v, 2)))

        elif k.startswith("n_obs_"):
            mlflow.set_tag(k, int(v))

    # -----------------------------
    # Modèle (pickle + artifact)
    # -----------------------------
    with open(model_path, "wb") as f:
        pickle.dump(last_model, f)

    mlflow.log_artifact(str(model_path), artifact_path="model")

print("Run MLflow loggé : params, métriques MAE, tags et modèle.")

🏃 View run Autoregressive_1_h12_bag1_conf1 at: http://127.0.0.1:5000/#/experiments/5/runs/bed08aef8ad749bd82f76ee4eccd4f6d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Run MLflow loggé : params, métriques MAE, tags et modèle.


# Régression linéaire

In [28]:
# ----------------------------
# Nixtla MLForecast
# ----------------------------
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ----------------------------
# Model backend
# ----------------------------
from sklearn.linear_model import LinearRegression

In [29]:
ts_lr["ds"] = (
    pd.to_datetime(ts_lr["ds"])
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

In [30]:
from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression

MLF_MODELS = {
    "LR_EXOG_ONLY": lambda freq: MLForecast(
        models={"LR": LinearRegression()},
        freq=freq,
        lags=[],               # ✅ pas de lags de y
        date_features=[],      # optionnel
    )
}

In [31]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from mlforecast.utils import PredictionIntervals

def _ensure_ms(x):
    """Force un Timestamp au 1er du mois (MS)."""
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp("start")

def _n_windows_monthly(ds_start, ds_end):
    """Nombre de mois inclusifs entre ds_start et ds_end."""
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def run_backtesting_h12_monthly(
    mlf,
    ts,
    *,
    h=12,
    # Expérience (dates des prédictions, i.e. la colonne `ds` dans le backtest)
    exp_start="1990-01-01",
    exp_end="2025-08-01",
    # Prévision mensuelle
    step_size=1,
    # Intervalles conformal
    pi_windows=24,
    levels=[95],
):
    ts = ts.copy()
    ts["ds"] = pd.to_datetime(ts["ds"])

    # 1) bornes exactes de l’expérience (sur les dates prédictibles)
    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    # 2) pour pouvoir prédire ds=exp_start à horizon h,
    #    il faut que le cutoff existe: cutoff = ds - h mois
    cutoff_start = exp_start - relativedelta(months=h)
    cutoff_end   = exp_end   - relativedelta(months=h)

    # 3) partitions = nombre de cutoffs mensuels (inclusif)
    partitions = _n_windows_monthly(cutoff_start, cutoff_end)

    # 4) (optionnel mais recommandé) filtrer le dataset à une plage utile
    #    On garde au minimum jusqu'à exp_end (features) et depuis assez tôt pour entraîner.
    ts = ts.loc[(ts["ds"] <= exp_end)].copy()

    # Prediction intervals (conformal)
    pi = PredictionIntervals(
        h=h,
        n_windows=pi_windows,
        method="conformal_distribution",
    )

    bkt_df = mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,     # ✅ ré-entrainement / cutoff chaque mois
        n_windows=partitions,    # ✅ nombre de partitions calculé automatiquement
        prediction_intervals=pi,
        level=levels,
        fitted=True,
        static_features=[],
    )

    meta = {
        "h": h,
        "step_size": step_size,
        "exp_start": exp_start,
        "exp_end": exp_end,
        "cutoff_start": cutoff_start,
        "cutoff_end": cutoff_end,
        "partitions": partitions,
        "pi_windows": pi_windows,
    }

    return bkt_df, meta

In [32]:
# ============================================================
# RUN – Linear Regression (Nixtla MLForecast) | EXOG-ONLY
# Mensuel (step=1) + horizon 12 + bornes exactes exp
# + sortie finale avec ds unique (dernier cutoff)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta

PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

# -----------------------------
# Paramètres
# -----------------------------
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")   # <- jusqu'à août 2025 inclus

def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start")

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

# -----------------------------
# Préparation des données
# -----------------------------
ts_lr = ts_lr.copy()

ts_lr["ds"] = (
    pd.to_datetime(ts_lr["ds"], errors="coerce")
      .values.astype("datetime64[M]").astype("datetime64[ns]")
)

if ts_lr["ds"].isna().any():
    bad = ts_lr[ts_lr["ds"].isna()].head()
    raise ValueError(f"Certaines dates 'ds' n'ont pas pu être parsées. Exemples:\n{bad}")

EXP_START = _ensure_ms(EXP_START)
EXP_END   = _ensure_ms(EXP_END)

CUTOFF_START = EXP_START - relativedelta(months=H)  # 1989-01-01
CUTOFF_END   = EXP_END   - relativedelta(months=H)  # 2024-08-01
PARTITIONS   = _n_windows_monthly(CUTOFF_START, CUTOFF_END)

print("✅ EXP ds range          :", EXP_START.date(), "→", EXP_END.date())
print("✅ CUTOFF range          :", CUTOFF_START.date(), "→", CUTOFF_END.date())
print("✅ PARTITIONS (n_windows):", PARTITIONS)
print("ts_lr ds range           :", ts_lr["ds"].min().date(), "→", ts_lr["ds"].max().date())

# optionnel (recommandé) : éviter fuite future
ts_lr = ts_lr[ts_lr["ds"] <= EXP_END].copy()

# -----------------------------
# Instancier le modèle
# -----------------------------
mlf = MLF_MODELS["LR_EXOG_ONLY"](FREQ)

model_names = list(mlf.models.keys())
first_name = next(iter(mlf.models))
print("Running models:", model_names)
print("First model class:", mlf.models[first_name].__class__.__name__)
print("Freq:", FREQ)

# -----------------------------
# Backtesting
# -----------------------------
bkt_lr, meta = run_backtesting_h12_monthly(
    mlf=mlf,
    ts=ts_lr,
    h=H,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
)

# Optionally reuse from meta:
EXP_START = meta["exp_start"]
EXP_END   = meta["exp_end"]

bkt_lr_eval = bkt_lr[(bkt_lr["ds"] >= EXP_START) & (bkt_lr["ds"] <= EXP_END)].copy()

# ============================================================
# ✅ 1 seule ligne par ds : garder le cutoff le plus récent
# ============================================================
bkt_lr_final = (
    bkt_lr_eval
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)                         # <- dernier cutoff (plus récent)
    .reset_index(drop=True)
)

# -----------------------------
# Checks
# -----------------------------
print("bkt_lr_eval rows         :", len(bkt_lr_eval))
print("bkt_lr_final rows        :", len(bkt_lr_final))
print("bkt_lr_final ds range    :", bkt_lr_final["ds"].min().date(), "→", bkt_lr_final["ds"].max().date())

# vérifie unicité
dup = bkt_lr_final.duplicated(subset=["unique_id", "ds"]).sum()
print("duplicates (unique_id, ds):", dup)

bkt_lr_final.head()

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA
✅ EXP ds range          : 1990-01-01 → 2025-08-01
✅ CUTOFF range          : 1989-01-01 → 2024-08-01
✅ PARTITIONS (n_windows): 428
ts_lr ds range           : 1960-01-01 → 2025-08-01
Running models: ['LR']
First model class: LinearRegression
Freq: MS
bkt_lr_eval rows         : 5070
bkt_lr_final rows        : 428
bkt_lr_final ds range    : 1990-01-01 → 2025-08-01
duplicates (unique_id, ds): 0


,unique_id,ds,cutoff,y,LR,LR-lo-95,LR-hi-95
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.227449,-0.350798,-0.104100
1,UNRATE,1990-02-01,1990-01-01,0.1,-0.373112,-1.363492,0.617268
2,UNRATE,1990-03-01,1990-02-01,0.2,-0.458527,-1.787253,0.870200
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.435854,-1.898439,1.026731
4,UNRATE,1990-05-01,1990-04-01,0.2,0.018847,-0.906300,0.943993


In [34]:
import pandas as pd

# --- détecter la colonne de prediction (ex: "LR") ---
pred_col = None
for c in bkt_lr_final.columns:
    if c not in {"unique_id", "ds", "cutoff", "y"} and c.startswith("LR"):
        # évite LR-lo-95 / LR-hi-95
        if "-lo-" not in c and "-hi-" not in c:
            pred_col = c
            break
if pred_col is None:
    # fallback: la colonne "LR" existe souvent
    pred_col = "LR"

# --- colonnes PI (si présentes) ---
lo_col = f"{pred_col}-lo-95"
hi_col = f"{pred_col}-hi-95"

cols = ["ds", "y", pred_col]
if lo_col in bkt_lr_final.columns and hi_col in bkt_lr_final.columns:
    cols += [lo_col, hi_col]

df_oos_lr = bkt_lr_final[cols].copy()
df_oos_lr = df_oos_lr.rename(columns={"ds": "date", "y": "y_true", pred_col: "y_hat"})
df_oos_lr["date"] = pd.to_datetime(df_oos_lr["date"])
df_oos_lr = df_oos_lr.set_index("date").sort_index()

if lo_col in df_oos_lr.columns and hi_col in df_oos_lr.columns:
    df_oos_lr = df_oos_lr.rename(columns={lo_col: "y_hat_p05", hi_col: "y_hat_p95"})

print(df_oos_lr.head())

            y_true     y_hat  y_hat_p05  y_hat_p95
date                                              
1990-01-01     0.0 -0.227449  -0.350798  -0.104100
1990-02-01     0.1 -0.373112  -1.363492   0.617268
1990-03-01     0.2 -0.458527  -1.787253   0.870200
1990-04-01     0.2 -0.435854  -1.898439   1.026731
1990-05-01     0.2  0.018847  -0.906300   0.943993


In [35]:
from sklearn.metrics import mean_absolute_error
import pandas as pd

periods = {
    "ALL":        (None, None),
    "1990_1999":  ("1990-01-01", "1999-12-31"),
    "2000_2008":  ("2000-01-01", "2008-12-31"),
    "2008_2019":  ("2008-01-01", "2019-12-31"),
    "2020_fin":   ("2020-01-01", None),
}

metrics_lr = {}

for label, (start, end) in periods.items():
    sub = df_oos_lr.copy()
    if start is not None:
        sub = sub[sub.index >= pd.Timestamp(start)]
    if end is not None:
        sub = sub[sub.index <= pd.Timestamp(end)]
    if len(sub) == 0:
        continue

    metrics_lr[f"mae_{label}"] = float(mean_absolute_error(sub["y_true"], sub["y_hat"]))
    metrics_lr[f"n_obs_{label}"] = int(len(sub))

print(metrics_lr)

{'mae_ALL': 0.8119687395195699, 'n_obs_ALL': 428, 'mae_1990_1999': 0.5097286591645158, 'n_obs_1990_1999': 120, 'mae_2000_2008': 0.46727185203509725, 'n_obs_2000_2008': 108, 'mae_2008_2019': 0.7975617172918482, 'n_obs_2008_2019': 144, 'mae_2020_fin': 1.8894670400339937, 'n_obs_2020_fin': 68}


In [54]:
import mlflow
import pickle
from pathlib import Path

# -----------------------------
# MODEL NAME
# -----------------------------
MODEL_NAME = "Linear Regression"

# -----------------------------
# (re)fit modèle final pour logging
# -----------------------------
mlf_final = MLF_MODELS["LR_EXOG_ONLY"](FREQ)
mlf_final.fit(
    ts_lr,
    id_col="unique_id",
    time_col="ds",
    target_col="y",
    static_features=[]  # mettre EXOG_COLS si nécessaire
)

# -----------------------------
# Params LR (config)
# -----------------------------
params_lr = {
    "model": MODEL_NAME,
    "horizon": H,
    "step_size": STEP_SIZE,
    "pi_windows": PI_WINDOWS,
    "levels": str(LEVELS),
    "freq": FREQ,
    "exp_start": str(EXP_START.date()),
    "exp_end": str(EXP_END.date()),
}

# -----------------------------
# Initialize MLflow
# -----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Test_MLFLOW")

run_name = f"{MODEL_NAME}_h{H}_step{STEP_SIZE}_piw{PI_WINDOWS}"

# -----------------------------
# Artifact path
# -----------------------------
TMP_DIR = Path("outputs/tmp_models")
TMP_DIR.mkdir(parents=True, exist_ok=True)

model_file_safe = MODEL_NAME.replace(" ", "_")
model_path = TMP_DIR / f"{model_file_safe}_mlf.pkl"

with mlflow.start_run(run_name=run_name):

    # -----------------------------
    # Tags
    # -----------------------------
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("end", str(EXP_END.date()))
    mlflow.set_tag("freq", str(FREQ))

    # -----------------------------
    # Params
    # -----------------------------
    for k, v in params_lr.items():
        if v is not None:
            mlflow.log_param(k, v)

    # -----------------------------
    # Metrics (MAE uniquement)
    # n_obs en tags
    # -----------------------------
    for k, v in metrics_lr.items():

        if v is None:
            continue

        if k.startswith("mae_"):
            mlflow.log_metric(k, float(round(v, 2)))

        elif k.startswith("n_obs_"):
            mlflow.set_tag(k, int(v))

    # -----------------------------
    # Modèle (pickle + artifact)
    # -----------------------------
    with open(model_path, "wb") as f:
        pickle.dump(mlf_final, f)

    mlflow.log_artifact(str(model_path), artifact_path="model")

print("Run MLflow loggé : params, métriques MAE, tags et modèle.")

🏃 View run Linear Regression_h12_step1_piw3 at: http://127.0.0.1:5000/#/experiments/5/runs/f7eb5846d8e24add8491a3bea3863db4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Run MLflow loggé : params, métriques MAE, tags et modèle.


# Calcul de la significativité

In [85]:
def make_mae_dm_pivot(
    wide: pd.DataFrame,
    segments: List[Tuple[str, Optional[str], str]],
    *,
    methods: Optional[Iterable[str]] = None,
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 20,
    round_digits: int = 4,
    add_dm: bool = True,
    dm_lags: int = 11,
    # >>> NEW
    show: str = "mae_p",          # "mae_p" (ancien), "p_only", "mae_only"
    p_digits: int = 3,
    best_cell: str = "-",         # affichage pour le best model / pas de test
    prefix_p: str = "",           # ex: "p=" si tu veux
) -> pd.DataFrame:

    df = wide.copy()

    # --- index datetime ---
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")  # pas de utc=True ici
        df = df.dropna(subset=["date"]).set_index("date")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("wide doit avoir un DatetimeIndex ou une colonne 'date'.")

    df = df.sort_index()

    # --- Harmonise TZ : si l'index est tz-naive, on garde tout tz-naive;
    #                    si tz-aware, on convertit les bornes en UTC.
    use_utc = df.index.tz is not None
    if use_utc:
        # si l'index est tz-aware mais pas UTC, on le convertit en UTC (robuste)
        df = df.tz_convert("UTC")

    if "true" not in df.columns:
        raise ValueError("wide doit contenir la colonne 'true'.")

    # méthodes
    if methods is None:
        meths = [c for c in df.columns if c != "true"]
    else:
        meths = [m for m in methods if m in df.columns and m != "true"]
    if len(meths) == 0:
        return pd.DataFrame()

    # fenêtres
    full_start, full_end = df.index.min(), df.index.max()
    windows: List[Tuple[pd.Timestamp, pd.Timestamp, str]] = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))

    for start, end, label in segments:
        if use_utc:
            s = pd.to_datetime(start, utc=True)
            e = pd.to_datetime(end, utc=True) if end is not None else full_end
        else:
            s = pd.to_datetime(start)
            e = pd.to_datetime(end) if end is not None else full_end
        windows.append((s, e, label))

    rows = []

    for start, end, label in windows:
        sub = df.loc[start:end, ["true"] + meths].copy().dropna(subset=["true"])

        maes: Dict[str, float] = {}
        err_abs: Dict[str, pd.Series] = {}

        for m in meths:
            diffs = (sub["true"] - sub[m]).abs().dropna()
            err_abs[m] = diffs
            maes[m] = float(diffs.mean()) if diffs.shape[0] >= min_obs else np.nan

        finite_models = [m for m in meths if isfinite(maes.get(m, np.nan))]
        best_m = min(finite_models, key=lambda k: maes[k]) if finite_models else None

        for m in meths:
            mae_val = maes.get(m, np.nan)
            if not isfinite(mae_val):
                rows.append((m, label, np.nan))
                continue

            # ----- affichage -----
            if show == "mae_only":
                cell = f"{mae_val:.{round_digits}f}"

            elif show == "p_only":
                if not add_dm or best_m is None or m == best_m:
                    cell = best_cell
                else:
                    v1, v2 = err_abs[m], err_abs[best_m]
                    common = v1.index.intersection(v2.index)
                    diff = (v1.loc[common] - v2.loc[common]).to_numpy()

                    if diff.size >= min_obs:
                        pval = dm_pvalue(diff, lags=dm_lags)
                        cell = f"{prefix_p}{pval:.{p_digits}f}" if isfinite(pval) else np.nan
                    else:
                        cell = np.nan

            else:  # "mae_p" (comportement ancien)
                cell = f"{mae_val:.{round_digits}f}"
                if add_dm and best_m is not None and m != best_m:
                    v1, v2 = err_abs[m], err_abs[best_m]
                    common = v1.index.intersection(v2.index)
                    diff = (v1.loc[common] - v2.loc[common]).to_numpy()

                    if diff.size >= min_obs:
                        pval = dm_pvalue(diff, lags=dm_lags)
                        if isfinite(pval):
                            cell = f"{cell} ({pval:.{p_digits}f})"

            rows.append((m, label, cell))

    out = pd.DataFrame(rows, columns=["model", "period", "value"])
    pivot = out.pivot(index="model", columns="period", values="value")

    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in segments]
    pivot = pivot.reindex(columns=desired_cols)
    pivot = pivot.reindex(index=meths)

    pivot.columns.name = "period"
    pivot.index.name = "model"
    return pivot


In [86]:
table_pivot_p = make_mae_dm_pivot(
    wide=wide,
    segments=segments,
    methods=methods_used,
    include_overall=True,
    overall_label="Ensemble",
    min_obs=20,
    add_dm=True,
    dm_lags=11,
    show="p_only",     # ✅ p-values seulement
    p_digits=3,
    best_cell="-",
    prefix_p="",       # ou "p=" si tu veux
)

print(table_pivot_p)


period            Ensemble 1990-1999 2000-2008 2008-2019 2020-fin
model                                                            
Autoregressive_1     0.369         -     0.341     0.523    0.455
Linear Regression        -     0.976         -         -        -


# Traking MLflow

In [89]:
import mlflow
import pickle
from pathlib import Path

# ============================================================
# Hypothèse :
# - table_pivot_p existe déjà (DataFrame p-values)
# - h, use_bagging, use_conformal, last_fit_end, y, params, metrics,
#   H, STEP_SIZE, PI_WINDOWS, EXP_END, FREQ, params_lr, metrics_lr,
#   last_model, mlf_final existent déjà
# ============================================================

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Test_MLFLOW_v2_clean")

TMP_DIR = Path("outputs/tmp_models")
TMP_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# ✅ UN SEUL artifact commun (HTML) -> lisible dans MLflow
# ============================================================

html_table = (
    table_pivot_p
    .style
    .set_properties(**{
        "font-size": "16px",
        "padding": "6px 10px",
        "text-align": "center"
    })
    .set_table_styles([
        {"selector": "th",
         "props": [("font-size", "16px"),
                   ("padding", "8px 10px"),
                   ("text-align", "center")]}
    ])
    .to_html()
)

table_html = TMP_DIR / "table_pivot_pvalues.html"
table_html.write_text(html_table, encoding="utf-8")

# ============================================================
# Runs modèles
# ============================================================
runs = [
    {
        "model_name": "Autoregressive_1",
        "run_name": f"Autoregressive_1_h{h}_bag{int(use_bagging)}_conf{int(use_conformal)}",
        "end": str(last_fit_end.date()) if last_fit_end is not None else "unknown",
        "freq": y.index.freqstr if getattr(y.index, "freq", None) is not None else "MS",
        "params": dict(params),
        "metrics": dict(metrics),
        "model_obj": last_model,
        "model_path": TMP_DIR / "Autoregressive_1_last_model.pkl",
    },
    {
        "model_name": "Linear Regression",
        "run_name": f"Linear_Regression_h{H}_step{STEP_SIZE}_piw{PI_WINDOWS}",
        "end": str(EXP_END.date()),
        "freq": str(FREQ),
        "params": dict(params_lr),
        "metrics": dict(metrics_lr),
        "model_obj": mlf_final,
        "model_path": TMP_DIR / "Linear_Regression_mlf.pkl",
    },
]

for cfg in runs:
    cfg["params"]["model"] = cfg["model_name"]

    with mlflow.start_run(run_name=cfg["run_name"]):

        # ---------------------------
        # Tags
        # ---------------------------
        mlflow.set_tag("model_name", cfg["model_name"])
        mlflow.set_tag("end", cfg["end"])
        mlflow.set_tag("freq", cfg["freq"])

        # ---------------------------
        # Params
        # ---------------------------
        for k, v in cfg["params"].items():
            if v is not None:
                mlflow.log_param(k, v)

        # ---------------------------
        # Metrics
        # ---------------------------
        for k, v in cfg["metrics"].items():
            if v is None:
                continue

            if k.startswith("mae_"):
                mlflow.log_metric(k, float(round(float(v), 2)))

            elif k.startswith("n_obs_"):
                mlflow.set_tag(k, str(int(v)))

            else:
                try:
                    mlflow.log_metric(k, float(v))
                except Exception:
                    mlflow.set_tag(k, str(v))

        # ====================================================
        # ✅ Artifact commun (HTML lisible)
        # ====================================================
        mlflow.log_artifact(str(table_html))

        # ---------------------------
        # Modèle
        # ---------------------------
        with open(cfg["model_path"], "wb") as f:
            pickle.dump(cfg["model_obj"], f)

        mlflow.log_artifact(str(cfg["model_path"]), artifact_path="model")

print("OK : table p-values HTML loggée dans chaque run.")

🏃 View run Autoregressive_1_h12_bag1_conf1 at: http://127.0.0.1:5000/#/experiments/6/runs/9a6f9847d05c4d7abd4a792ce00d5764
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
🏃 View run Linear_Regression_h12_step1_piw3 at: http://127.0.0.1:5000/#/experiments/6/runs/ec0aa3d93b584796b65461c3f2394e55
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
OK : table p-values HTML loggée dans chaque run.
